# Feature Engineering

> Config

> Input - Carga de datos

> Utils

> FE

    - A. Filtros (media móvil y pasa-altos)

    - B. Transformaciones FFT en ventanas previas

    - C. Deltas de frecuencias

    - D. Lags y otros deltas

> Output - Export raw dataset

### Config

- Ruta dataset detectada: `../inputs/dataset_v2.csv`  
- Features pre-seleccionadas del análisis descriptivo

In [1]:
import os
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

# Ruta del dataset (ajustá si es necesario):
DATA_PATH = r"""../inputs/dataset_v2.csv"""

# Columna de fecha
DATE_COL = "fecha"

# Rando de fechas a utilizar:
DATE_START = "2023-01-01"
DATE_END = "2025-09-30"

# Variable objetivo y horizonte de forecast (ejemplo: retorno del MERVAL t+1)
TARGET_COL = "merval_apertura"
H = 1   # horizonte en pasos

# Lista de features pre-seleccionadas
SELECTED_FEATURES = [
    "emae_original", # "emae_desestacionalizado", 
    # BADIAR:
    "badiar_total",             # o "badiar_num" / el que tengas en tu dataset
    # Tipo de cambio:
    "tc_mayorista", "tc_minorista",
    # Agregados / circulación monetaria:
    "base_monetaria", "m1", "m2", "m2_transaccional",
    # Préstamos (en pesos) y tasas:
    "prestamos_privados_ars", "tasa_prestamos_personales",
    # Inflación:
    "inflacion"                 # o "ipc_general" / "ipc_nucleo" según tu CSV
]


# Opcional: columnas a excluir (IDs, texto libre, etc.)
EXCLUDE_COLS = []

pd.options.display.max_columns = None
pd.options.display.width = None

### Input - Carga de datos

In [2]:
# Intentamos leer parquet/csv en orden. Ajustá DATA_PATH según tu proyecto.
df = None
if os.path.exists(DATA_PATH):
    if DATA_PATH.lower().endswith(".parquet"):
        df = pd.read_parquet(DATA_PATH)
    elif DATA_PATH.lower().endswith(".csv"):
        df = pd.read_csv(DATA_PATH)
    else:
        try:
            df = pd.read_parquet(DATA_PATH)
        except Exception:
            df = pd.read_csv(DATA_PATH)
else:
    print(f"[ADVERTENCIA] No se encontró DATA_PATH: {DATA_PATH}")

# Si hay columna de fecha, la usamos como índice temporal
if df is not None and DATE_COL and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.sort_values(DATE_COL).set_index(DATE_COL)

# Filtramos por rango de fechas si se especifica
if df is not None and DATE_START and DATE_END:
    df = df.loc[(df.index >= DATE_START) & (df.index <= DATE_END)]

print("Forma del df:", None if df is None else df.shape)
print("Columnas:", None if df is None else list(df.columns)[:10], "...")

Forma del df: (682, 36)
Columnas: ['merval_apertura', 'merval_maximo', 'merval_minimo', 'merval_cierre', 'embi_spread_arg', 'embi_spreads_brz', 'embi_spread_global', 'emae_original', 'emae_desestacionalizado', 'emae_tendencia_ciclo'] ...


In [3]:
df.index.min()

Timestamp('2023-01-02 00:00:00')

In [4]:
df.index.max()

Timestamp('2025-09-30 00:00:00')

In [5]:
df.isna().sum()

merval_apertura                     0
merval_maximo                       0
merval_minimo                       0
merval_cierre                       0
embi_spread_arg                    42
embi_spreads_brz                   42
embi_spread_global                 42
emae_original                      23
emae_desestacionalizado            23
emae_tendencia_ciclo               23
tc_minorista                       16
tc_mayorista                       16
rrii                               16
tamar                             441
badlar                             16
tm20                               16
tasa_prestamos_personales          16
tasa_pf_pesos                      16
tasa_pf_dolares                    16
base_monetaria                     16
circulacion_monetaria              16
dep_ccorrientes                    16
dep_cahorro                        16
dep_plazo                          16
dep_plazo_fijo                     16
m1                                 16
m2          

### Utils
Funciones auxiliares para filtros, FFT por ventanas, fractales y lags.

In [6]:
import math

def ensure_series(x):
    return x.dropna().astype(float)

def moving_average(s, window: int = 5, min_periods: int = 1):
    s = ensure_series(s)
    return s.rolling(window=window, min_periods=min_periods).mean()

def high_pass_via_ema(s, span: int = 20):
    """
    Alta-paso simple: quita la componente de baja frecuencia restando la EMA.
    """
    s = ensure_series(s)
    ema = s.ewm(span=span, adjust=False).mean()
    return s - ema

def high_pass_via_rolling_mean(s, window: int = 20):
    s = ensure_series(s)
    ma = s.rolling(window=window, min_periods=1).mean()
    return s - ma

def pct_change_safe(s, periods: int = 1):
    s = ensure_series(s)
    return s.pct_change(periods=periods).replace([np.inf, -np.inf], np.nan)

def diff_safe(s, periods: int = 1):
    s = ensure_series(s)
    return s.diff(periods=periods)

def rolling_std(s, window: int = 20, min_periods: int = 5):
    s = ensure_series(s)
    return s.rolling(window=window, min_periods=min_periods).std()


def windowed_fft_features(s: pd.Series, window: int = 14, step: int = 1, bands=None):
    """
    Calcula features en el dominio de la frecuencia en ventanas deslizantes.
    Devuelve un DataFrame indexado por el mismo índice temporal.
    - 'dom_freq': frecuencia dominante (índice normalizado)
    - 'spec_centroid': centroide espectral
    - 'spec_entropy': entropía de potencia normalizada
    - 'band_power_[i]': potencia por banda si bands está definido
    """
    s = ensure_series(s)
    x = s.values
    idx = s.index

    if bands is None:
        bands = [(1, 2), (3, 6), (7, 12), (13, 24)]  # ejemplo en bins

    feats = {
        "dom_freq": [],
        "spec_centroid": [],
        "spec_entropy": [],
    }
    for i, _ in enumerate(bands):
        feats[f"band_power_{i}"] = []

    out_index = []
    N = len(x)
    for start in range(0, N - window + 1, step):
        seg = x[start : start + window]
        seg = seg - np.nanmean(seg)
        mag = np.abs(np.fft.rfft(seg))
        mag = np.nan_to_num(mag, nan=0.0, posinf=0.0, neginf=0.0)

        if len(mag) > 1:
            dom_bin = int(np.argmax(mag[1:])) + 1
        else:
            dom_bin = 0

        bins = np.arange(len(mag))
        power = mag ** 2
        power_sum = power.sum() + 1e-12
        centroid = float((bins * power).sum() / power_sum)

        p = power / power_sum
        p = p + 1e-12
        entropy = float(-(p * np.log(p)).sum() / np.log(len(p)))

        feats["dom_freq"].append(dom_bin)
        feats["spec_centroid"].append(centroid)
        feats["spec_entropy"].append(entropy)

        for i, (b0, b1) in enumerate(bands):
            b0 = max(0, int(b0))
            b1 = min(len(mag) - 1, int(b1))
            feats[f"band_power_{i}"].append(float(power[b0 : b1 + 1].sum()))

        out_index.append(idx[start + window - 1])

    out = pd.DataFrame(feats, index=pd.Index(out_index, name=s.index.name))
    return out

def make_lagged(df, cols, lags=(1, 5, 10, 20)):
    out = {}
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c]
        for L in lags:
            out[f"{c}_lag{L}"] = s.shift(L)
    return pd.DataFrame(out, index=df.index)

def make_deltas(df, cols, periods=(1, 5, 10)):
    out = {}
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c]
        for p in periods:
            out[f"{c}_diff{p}"] = diff_safe(s, p)
            out[f"{c}_pct{p}"] = pct_change_safe(s, p)
    return pd.DataFrame(out, index=df.index)

### Feature Engineering

#### A. Filtros (media móvil y pasa-altos)

In [7]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    base_cols = [c for c in SELECTED_FEATURES if c in df.columns]
    fe_ma = pd.DataFrame(index=df.index)
    fe_hp = pd.DataFrame(index=df.index)

    for c in base_cols:
        fe_ma[f"{c}_ma_3"] = moving_average(df[c], 3)
        fe_ma[f"{c}_ma_5"] = moving_average(df[c], 5)
        fe_hp[f"{c}_hp_ema20"] = high_pass_via_ema(df[c], span=20)
        fe_hp[f"{c}_hp_rm20"] = high_pass_via_rolling_mean(df[c], window=20)

    FE_FILTERS = pd.concat([fe_ma, fe_hp], axis=1)
    print("FE_FILTERS shape:", FE_FILTERS.shape)


FE_FILTERS shape: (682, 40)


In [8]:
FE_FILTERS.describe(include='all')

,emae_original_ma_3,emae_original_ma_5,tc_mayorista_ma_3,tc_mayorista_ma_5,tc_minorista_ma_3,tc_minorista_ma_5,base_monetaria_ma_3,base_monetaria_ma_5,m1_ma_3,m1_ma_5,m2_ma_3,m2_ma_5,m2_transaccional_ma_3,m2_transaccional_ma_5,prestamos_privados_ars_ma_3,prestamos_privados_ars_ma_5,tasa_prestamos_personales_ma_3,tasa_prestamos_personales_ma_5,inflacion_ma_3,inflacion_ma_5,emae_original_hp_ema20,emae_original_hp_rm20,tc_mayorista_hp_ema20,tc_mayorista_hp_rm20,tc_minorista_hp_ema20,tc_minorista_hp_rm20,base_monetaria_hp_ema20,base_monetaria_hp_rm20,m1_hp_ema20,m1_hp_rm20,m2_hp_ema20,m2_hp_rm20,m2_transaccional_hp_ema20,m2_transaccional_hp_rm20,prestamos_privados_ars_hp_ema20,prestamos_privados_ars_hp_rm20,tasa_prestamos_personales_hp_ema20,tasa_prestamos_personales_hp_rm20,inflacion_hp_ema20,inflacion_hp_rm20
count,659.000000,659.000000,666.000000,666.000000,666.000000,666.000000,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.670000e+02,6.670000e+02,666.000000,666.000000,682.000000,682.000000,659.000000,659.000000,666.000000,666.000000,666.000000,666.000000,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.660000e+02,6.670000e+02,6.670000e+02,666.000000,666.000000,682.000000,682.000000
mean,148.542280,148.530442,760.370623,758.629990,786.008646,784.227592,1.848429e+07,1.842999e+07,2.307501e+07,2.302420e+07,4.276586e+07,4.266854e+07,2.857894e+07,2.851490e+07,4.072419e+07,4.057687e+07,87.919449,87.924574,6.653519,6.659238,0.119885,0.112463,17.150460,17.320240,17.433196,17.626294,5.200017e+05,5.155223e+05,4.783036e+05,4.775458e+05,8.973875e+05,8.974701e+05,6.018822e+05,6.007675e+05,1.387451e+06,1.396374e+06,-0.044924,-0.022250,-0.054610,-0.054326
std,6.379924,6.336296,385.023511,384.848953,391.787361,391.653472,1.227126e+07,1.224268e+07,1.230768e+07,1.230064e+07,2.231792e+07,2.230195e+07,1.553290e+07,1.551677e+07,3.161014e+07,3.153389e+07,23.791698,23.711025,5.387672,5.367418,2.688231,3.204300,37.995068,44.674556,38.217362,45.119338,1.081217e+06,1.208988e+06,6.063897e+05,6.946547e+05,1.433920e+06,1.656162e+06,1.282381e+06,1.465366e+06,1.052400e+06,1.086834e+06,6.160420,6.985535,1.553607,1.805986
min,133.637222,133.637222,178.140000,178.140000,185.360000,185.360000,5.082551e+06,5.092677e+06,7.320239e+06,7.330173e+06,1.211828e+07,1.218961e+07,8.826307e+06,8.853073e+06,7.384489e+06,7.391233e+06,59.900000,60.714000,1.500000,1.500000,-7.582736,-8.940875,-66.880464,-75.096000,-61.432474,-68.214500,-2.219103e+06,-2.738321e+06,-1.347566e+06,-1.538033e+06,-3.265696e+06,-3.368745e+06,-2.323925e+06,-3.254330e+06,-1.732360e+05,-2.842835e+05,-32.522917,-36.273000,-7.052586,-7.030000
25%,146.294126,145.655740,349.990833,349.991000,366.904167,366.898000,6.473939e+06,6.469774e+06,1.042194e+07,1.039672e+07,1.953664e+07,1.952134e+07,1.295730e+07,1.291238e+07,1.278881e+07,1.278431e+07,69.918333,69.736500,2.625000,2.610000,-1.210223,-1.408232,5.811263,5.539375,5.670026,5.333125,-1.626581e+04,-2.693566e+04,1.153801e+05,9.148615e+04,5.684235e+04,-3.587787e+03,-1.450926e+05,-2.595917e+05,4.630636e+05,4.558465e+05,-1.728173,-1.800375,-0.409435,-0.423750
50%,148.401054,148.275502,886.750000,886.100000,927.716667,927.150000,1.487172e+07,1.494677e+07,2.089785e+07,2.086297e+07,4.215237e+07,4.166918e+07,2.521764e+07,2.504991e+07,2.720129e+07,2.701511e+07,79.110000,78.753000,4.600000,4.600000,-0.321655,-0.116703,8.422079,8.115000,8.117176,7.835500,2.043598e+05,2.087327e+05,3.551655e+05,3.418237e+05,5.688641e+05,5.738694e+05,2.462938e+05,3.162093e+05,1.173689e+06,1.161810e+06,0.038983,0.049500,-0.017545,0.000000
75%,152.557032,152.251702,1045.910833,1044.759000,1071.428333,1070.720000,2.985432e+07,2.989981e+07,3.568231e+07,3.560823e+07,6.360445e+07,6.374197e+07,4.483305e+07,4.475402e+07,6.684950e+07,6.649187e+07,102.937500,102.634000,8.400000,8.400000,0.956756,0.945413,10.425600,9.693625,10.474584,10.090875,8.046465e+05,8.819905e+05,7.931486e+05,7.713720e+05,1.5

#### B. Transformaciones FFT en ventanas previas

In [9]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    base_cols = [c for c in SELECTED_FEATURES if c in df.columns]
    fft_feats = []
    for c in base_cols:
        fdf = windowed_fft_features(df[c], window=28, step=1,
                                    bands=[(1,3),(4,7),(8,14)])
        fdf = fdf.add_prefix(f"{c}_")
        fft_feats.append(fdf)
    FE_FFT = pd.concat(fft_feats, axis=1) if fft_feats else pd.DataFrame(index=df.index)
    print("FE_FFT shape:", FE_FFT.shape)


FE_FFT shape: (655, 60)


In [10]:
FE_FFT.describe(include='all')

,emae_original_dom_freq,emae_original_spec_centroid,emae_original_spec_entropy,emae_original_band_power_0,emae_original_band_power_1,emae_original_band_power_2,tc_mayorista_dom_freq,tc_mayorista_spec_centroid,tc_mayorista_spec_entropy,tc_mayorista_band_power_0,tc_mayorista_band_power_1,tc_mayorista_band_power_2,tc_minorista_dom_freq,tc_minorista_spec_centroid,tc_minorista_spec_entropy,tc_minorista_band_power_0,tc_minorista_band_power_1,tc_minorista_band_power_2,base_monetaria_dom_freq,base_monetaria_spec_centroid,base_monetaria_spec_entropy,base_monetaria_band_power_0,base_monetaria_band_power_1,base_monetaria_band_power_2,m1_dom_freq,m1_spec_centroid,m1_spec_entropy,m1_band_power_0,m1_band_power_1,m1_band_power_2,m2_dom_freq,m2_spec_centroid,m2_spec_entropy,m2_band_power_0,m2_band_power_1,m2_band_power_2,m2_transaccional_dom_freq,m2_transaccional_spec_centroid,m2_transaccional_spec_entropy,m2_transaccional_band_power_0,m2_transaccional_band_power_1,m2_transaccional_band_power_2,prestamos_privados_ars_dom_freq,prestamos_privados_ars_spec_centroid,prestamos_privados_ars_spec_entropy,prestamos_privados_ars_band_power_0,prestamos_privados_ars_band_power_1,prestamos_privados_ars_band_power_2,tasa_prestamos_personales_dom_freq,tasa_prestamos_personales_spec_centroid,tasa_prestamos_personales_spec_entropy,tasa_prestamos_personales_band_power_0,tasa_prestamos_personales_band_power_1,tasa_prestamos_personales_band_power_2,inflacion_dom_freq,inflacion_spec_centroid,inflacion_spec_entropy,inflacion_band_power_0,inflacion_band_power_1,inflacion_band_power_2
count,632.000000,632.000000,632.000000,632.000000,632.000000,632.000000,639.000000,639.000000,639.000000,6.390000e+02,6.390000e+02,6.390000e+02,639.000000,639.000000,639.000000,6.390000e+02,6.390000e+02,639.000000,639.000000,639.000000,639.000000,6.390000e+02,6.390000e+02,6.390000e+02,639.000000,639.000000,639.000000,6.390000e+02,6.390000e+02,6.390000e+02,639.000000,639.000000,639.000000,6.390000e+02,6.390000e+02,6.390000e+02,639.000000,639.000000,639.000000,6.390000e+02,6.390000e+02,6.390000e+02,640.000000,640.000000,640.000000,6.400000e+02,6.400000e+02,6.400000e+02,639.000000,639.000000,639.000000,639.000000,639.000000,639.000000,655.000000,6.550000e+02,6.550000e+02,6.550000e+02,6.550000e+02,6.550000e+02
mean,1.162975,2.411032,0.475594,2650.239075,306.873494,194.436776,1.156495,2.643472,0.544339,5.712992e+05,7.758331e+04,4.426481e+04,1.286385,2.752208,0.555257,5.988524e+05,6.656009e+04,31405.944882,1.505477,3.122077,0.578545,4.476008e+14,8.907813e+13,5.746069e+13,1.225352,2.850119,0.563238,1.779673e+14,2.301702e+13,1.566447e+13,1.222222,2.567549,0.520344,9.244575e+14,1.208809e+14,6.337633e+13,1.253521,2.231347,0.448662,7.138557e+14,6.863517e+13,3.934398e+13,1.128125,2.339745,0.491365,7.957523e+14,9.253769e+13,5.857380e+13,2.594679,4.490294,0.681334,12099.488811,2186.598230,1770.968455,1.155725,2.385842e+00,4.676027e-01,8.369689e+02,1.077645e+02,6.714153e+01
std,0.585365,0.944351,0.155583,4338.239163,479.758077,294.269266,0.815553,0.980314,0.106456,2.504893e+06,2.903697e+05,1.770598e+05,1.598683,1.158939,0.118247,2.655755e+06,2.332115e+05,110871.390896,1.303318,1.185035,0.157512,6.261755e+14,1.734135e+14,1.100145e+14,0.705497,0.772929,0.128366,2.585927e+14,2.535688e+13,1.674469e+13,0.519893,0.762072,0.146642,1.175241e+15,1.267336e+14,7.696635e+13,0.532530,0.735970,0.157314,8.292106e+14,9.655310e+13,5.608951e+13,0.334490,0.614305,0.163530,7.434684e+14,9.782787e+13,6.016967e+13,2.758294,1.532554,0.126807,21288.267048,2921.398895,2386.663981,0.533461,1.064594e+00,1.775217e-01,2.173599e+03,2.825229e+02,1.710903e+02
min,1.000000,1.759778,0.275970,0.319941,0.026348,0.019591,1.000000,1.496029,0.239341,1.325862e-02,3.457336e-03,2.036044e-02,1.000000,1.500461,0.225497,2.974858e+00,3.775822e+00,3.393100,1.000000,1.341910,0.153062,4.342848e+11,1.058870e+11,8.622824e+10,1.000000,1.405852,0.170909,5.066025e+12,1.443833e+11,2.299952e+11,1.000000,1.392058,0.177614,1.477283e+13,6.862792e+11,7.277969

#### C. Deltas de frecuencias

In [11]:
FE_FFT.empty

False

In [12]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    fe_freq_delta = pd.DataFrame(df.index)
    if not FE_FFT.empty:
        for col in FE_FFT.columns:
            if col.endswith("band_power_0"):
                base = col.rsplit("_band_power_", 1)[0]
                b0 = FE_FFT[f"{base}_band_power_0"]
                b1 = FE_FFT[f"{base}_band_power_1"]
                b2 = FE_FFT[f"{base}_band_power_2"]
                fe_freq_delta[f"{base}_bp01"] = b0 - b1
                fe_freq_delta[f"{base}_bp12"] = b1 - b2

    FE_FREQ_DELTAS = fe_freq_delta
    print("FE_FREQ_DELTAS shape:", FE_FREQ_DELTAS.shape)


FE_FREQ_DELTAS shape: (682, 21)


In [13]:
FE_FREQ_DELTAS.describe(include='all')

,fecha,emae_original_bp01,emae_original_bp12,tc_mayorista_bp01,tc_mayorista_bp12,tc_minorista_bp01,tc_minorista_bp12,base_monetaria_bp01,base_monetaria_bp12,m1_bp01,m1_bp12,m2_bp01,m2_bp12,m2_transaccional_bp01,m2_transaccional_bp12,prestamos_privados_ars_bp01,prestamos_privados_ars_bp12,tasa_prestamos_personales_bp01,tasa_prestamos_personales_bp12,inflacion_bp01,inflacion_bp12
count,682,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,2024-05-19 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,2023-01-02 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2023-09-11 06:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2024-05-22 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2025-01-22 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,2025-09-30 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
FE_FFT.columns

Index(['emae_original_dom_freq', 'emae_original_spec_centroid',
       'emae_original_spec_entropy', 'emae_original_band_power_0',
       'emae_original_band_power_1', 'emae_original_band_power_2',
       'tc_mayorista_dom_freq', 'tc_mayorista_spec_centroid',
       'tc_mayorista_spec_entropy', 'tc_mayorista_band_power_0',
       'tc_mayorista_band_power_1', 'tc_mayorista_band_power_2',
       'tc_minorista_dom_freq', 'tc_minorista_spec_centroid',
       'tc_minorista_spec_entropy', 'tc_minorista_band_power_0',
       'tc_minorista_band_power_1', 'tc_minorista_band_power_2',
       'base_monetaria_dom_freq', 'base_monetaria_spec_centroid',
       'base_monetaria_spec_entropy', 'base_monetaria_band_power_0',
       'base_monetaria_band_power_1', 'base_monetaria_band_power_2',
       'm1_dom_freq', 'm1_spec_centroid', 'm1_spec_entropy', 'm1_band_power_0',
       'm1_band_power_1', 'm1_band_power_2', 'm2_dom_freq', 'm2_spec_centroid',
       'm2_spec_entropy', 'm2_band_power_0', 'm2_band_

In [15]:
FE_FREQ_DELTAS

,fecha,emae_original_bp01,emae_original_bp12,tc_mayorista_bp01,tc_mayorista_bp12,tc_minorista_bp01,tc_minorista_bp12,base_monetaria_bp01,base_monetaria_bp12,m1_bp01,m1_bp12,m2_bp01,m2_bp12,m2_transaccional_bp01,m2_transaccional_bp12,prestamos_privados_ars_bp01,prestamos_privados_ars_bp12,tasa_prestamos_personales_bp01,tasa_prestamos_personales_bp12,inflacion_bp01,inflacion_bp12
0,2023-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677,2025-09-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
678,2025-09-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
679,2025-09-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
680,2025-09-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### D. Lags y otros deltas

In [16]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    base_cols = [c for c in SELECTED_FEATURES if c in df.columns]
    FE_LAGS = make_lagged(df, base_cols, lags=(1,2,3))
    FE_DELTAS = make_deltas(df, base_cols, periods=(1,3,10))
    FE_ROLLSTD = pd.DataFrame({
        f"{c}_std20": rolling_std(df[c], window=20)
        for c in base_cols
    })
    print("FE_LAGS:", FE_LAGS.shape, "FE_DELTAS:", FE_DELTAS.shape, "FE_ROLLSTD:", FE_ROLLSTD.shape)


FE_LAGS: (682, 30) FE_DELTAS: (682, 60) FE_ROLLSTD: (682, 10)


In [17]:
FE_LAGS

,emae_original_lag1,emae_original_lag2,emae_original_lag3,tc_mayorista_lag1,tc_mayorista_lag2,tc_mayorista_lag3,tc_minorista_lag1,tc_minorista_lag2,tc_minorista_lag3,base_monetaria_lag1,base_monetaria_lag2,base_monetaria_lag3,m1_lag1,m1_lag2,m1_lag3,m2_lag1,m2_lag2,m2_lag3,m2_transaccional_lag1,m2_transaccional_lag2,m2_transaccional_lag3,prestamos_privados_ars_lag1,prestamos_privados_ars_lag2,prestamos_privados_ars_lag3,tasa_prestamos_personales_lag1,tasa_prestamos_personales_lag2,tasa_prestamos_personales_lag3,inflacion_lag1,inflacion_lag2,inflacion_lag3
fecha,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-03,143.03019,NaN,NaN,178.14,NaN,NaN,185.36,NaN,NaN,5200397.0,NaN,NaN,7515263.0,NaN,NaN,12877548.0,NaN,NaN,9.643075e+06,NaN,NaN,7.578639e+06,NaN,NaN,83.42,NaN,NaN,6.0,NaN,NaN
2023-01-04,143.03019,143.03019,NaN,178.43,178.14,NaN,185.99,185.36,NaN,5219463.0,5200397.0,NaN,7495631.0,7515263.0,NaN,12720840.0,12877548.0,NaN,9.544024e+06,9.643075e+06,NaN,7.542830e+06,7.578639e+06,NaN,83.46,83.42,NaN,6.0,6.0,NaN
2023-01-05,143.03019,143.03019,143.03019,178.65,178.43,178.14,186.22,185.99,185.36,5247080.0,5219463.0,5200397.0,7521155.0,7495631.0,7515263.0,12670683.0,12720840.0,12877548.0,9.587668e+06,9.544024e+06,9.643075e+06,7.472713e+06,7.542830e+06,7.578639e+06,84.59,83.46,83.42,6.0,6.0,6.0
2023-01-06,143.03019,143.03019,143.03019,178.94,178.65,178.43,186.16,186.22,185.99,5234491.0,5247080.0,5219463.0,7415678.0,7521155.0,7495631.0,12636832.0,12670683.0,12720840.0,9.604629e+06,9.587668e+06,9.544024e+06,7.456385e+06,7.472713e+06,7.542830e+06,83.91,84.59,83.46,6.0,6.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-24,NaN,NaN,NaN,1354.17,1425.67,1474.75,1391.95,1438.30,1523.75,41199281.0,41976631.0,40711384.0,40970642.0,40931544.0,40249157.0,74969859.0,74642745.0,73809873.0,5.034691e+07,5.055832e+07,5.050484e+07,1.050277e+08,1.059497e+08,1.063919e+08,80.99,84.15,81.81,2.1,2.1,2.1
2025-09-25,NaN,NaN,NaN,1343.33,1354.17,1425.67,1367.95,1391.95,1438.30,41246514.0,41199281.0,41976631.0,40574923.0,40970642.0,40931544.0,75093671.0,74969859.0,74642745.0,5.063649e+07,5.034691e+07,5.055832e+07,1.050253e+08,1.050277e+08,1.059497e+08,80.88,80.99,84.15,2.1,2.1,2.1
2025-09-26,NaN,NaN,NaN,1325.08,1343.33,1354.17,1354.03,1367.95,1391.95,40742575.0,41246514.0,41199281.0,40845843.0,40574923.0,40970642.0,75900697.0,75093671.0,74969859.0,5.109785e+07,5.063649e+07,5.034691e+07,1.050022e+08,1.050253e+08,1.050277e+08,81.04,80.88,80.99,2.1,2.1,2.1


#### Output - Ensamble y Exportación de Features

In [ ]:
if df is None:
    print("[INFO] Saltando: no hay df cargado.")
else:
    parts = []
    for name in ["FE_FILTERS","FE_FFT","FE_FREQ_DELTAS","FE_FRACTAL","FE_LAGS","FE_DELTAS","FE_ROLLSTD"]:
        if name in globals():
            part = globals()[name]
            if isinstance(part, pd.DataFrame) and not part.empty:
                parts.append(part)
    FE_ALL = pd.concat(parts, axis=1).sort_index() if parts else pd.DataFrame(index=df.index)

    if TARGET_COL in df.columns:
        y = df[TARGET_COL].shift(-H)
        FE_ALL = FE_ALL.join(y.rename(f"{TARGET_COL}_t_plus_{H}"))
    FE_ALL = FE_ALL.dropna(how="any")

    print("FE_ALL shape:", FE_ALL.shape)
    out_path = "/mnt/data/features_engineered.parquet"
    try:
        FE_ALL.to_parquet(out_path)
        print(f"Guardado: {out_path}")
    except Exception as e:
        print("[WARN] No se pudo guardar en parquet:", e)
        csv_path = "/mnt/data/features_engineered.csv"
        FE_ALL.to_csv(csv_path, index=True)
        print(f"Guardado CSV: {csv_path}")
